ESPECTRO DE ENERGÍAS $E(V_z)$ y transición de fase topológica

Modelo para entender la superconductividad topológica unidimensional mediante una cadena de fermiones de espin nulo.

Para ello se programa el hamiltoniano de la cadena de sitio o potencial químico ($\mu$), hopping ($t$) y emparejamiento superconductor de onda p (\Delta). Para resolver el hamiltoniano en segunda cuantización, se usa la base de Nambu (la dimension de la matriz es 4N $\times$ 4N) o de operadores fermiónicos para una cadena de $N$ sitios. El sistema se resuelve diagonalizando la matriz y se grafican los autovalores. Se observa que en la fase topológica ($ t = 0, \mu = 0 $) aparece un par de estados degenerados con energía nula que corresponden a los MZMs.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Parámetros físicos y de red ---
N = 50  # Número de sitios en la cadena
t = 1.0  # Hopping entre primeros vecinos
mu = -2.0 * t  # Potencial químico (alineado para Vz_crit = Delta)
delta = 1.0  # Parámetro de orden superconductor
alpha = 2.0  # Acoplamiento espín-órbita (Rashba)

# --- 2. Matrices de Pauli y bases de Nambu ---
sigma_0 = np.eye(2)
sigma_x = np.array([[0, 1], [1, 0]])
sigma_y = np.array([[0, -1j], [1j, 0]])
sigma_z = np.array([[1, 0], [0, -1]])

tau_0, tau_x, tau_y, tau_z = sigma_0, sigma_x, sigma_y, sigma_z

# Productos de Kronecker (espacio Nambu x Espín)
tz_s0 = np.kron(tau_z, sigma_0)
t0_sz = np.kron(tau_0, sigma_z)
tx_s0 = np.kron(tau_x, sigma_0)
tz_sy = np.kron(tau_z, sigma_y)


# --- 3. Definición de bloques del Hamiltoniano ---
def H_site(Vz):
    return -mu * tz_s0 + Vz * t0_sz + delta * tx_s0


def H_hop():
    return -t * tz_s0 - 1j * (alpha / 2.0) * tz_sy


def build_BdG_matrix(Vz):
    H = np.zeros((4 * N, 4 * N), dtype=complex)
    Hs = H_site(Vz)
    Hh = H_hop()

    for i in range(N):
        # Término local
        H[4 * i : 4 * i + 4, 4 * i : 4 * i + 4] = Hs
        # Término de salto a primeros vecinos
        if i < N - 1:
            H[4 * i : 4 * i + 4, 4 * (i + 1) : 4 * (i + 1) + 4] = Hh
            H[4 * (i + 1) : 4 * (i + 1) + 4, 4 * i : 4 * i + 4] = Hh.conj().T

    return H


# --- 4. Diagonalización vs Vz ---
Vz_values = np.linspace(0.0, 3.0, 150)
energies = []

for Vz in Vz_values:
    H_BdG = build_BdG_matrix(Vz)
    evals = np.linalg.eigvalsh(H_BdG)
    energies.append(evals)

energies = np.array(energies)

# --- 5. Gráfica del espectro ---
plt.figure(figsize=(8, 5))
plt.plot(Vz_values, energies, "k-", alpha=0.3, linewidth=0.8)
plt.axvline(
    x=1.0, color="r", linestyle="--", label=r"Punto crítico $V_z = \Delta = 1.0$"
)
plt.xlabel(r"Campo Zeeman $V_z$")
plt.ylabel(r"Energía $E$")
plt.title(
    r"Espectro de energía BdG vs $V_z$ ($\mu = -2t$, $\Delta = 1.0$, $\alpha = 2.0$)"
)
plt.ylim(-2.5, 2.5)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

KeyboardInterrupt: 

Localización espacial $\lvert \psi_i \rvert^2$ (Modos ligados de Majorana) a través de una simulación de un hilo semiconductor con Acoplamiento de Rashba (modelo de Lutchyn-Oreg). 

Este modelo de juguete simula una implementación realista de hilos semiconductores (como InAs o InSb) con fuerte acoplamiento $LS$ fuerte acoplados a un superconductor convencional de onda $s$. El código resuelve el espectro de BdG considerando la energía cinética, el acoplamiento Rashba ($\alpha$) el campo magnético de Zeeman ($E_z$) y la superconductividad inducida ($\Delta$). Para implementarlo, se construye el hamiltoniano BdG en la respresentación de momentos o mediante discretización en el espacio real (diferencias finitas). Con el hamiltoniano construido, se evalúa el espectro de energías en función del campo magnético magnético aplicado para identificar la transición de fase topológica, la cual ocurrecuando el campo Zeeman supera el umbral crítico dado por $E_z^c = \sqrt{\Delta^2 + \mu^2}$. Con este código se demuestra el cierre y posterior reapertura de la brecha de energía, donde el sistema entra en "fase topológica" y aparecen modos ligados de Majorana en los extremos en lso bordes del hilo.



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Parámetros compartidos
N = 50
t = 1.0
mu = -2.0 * t
delta = 1.0
alpha = 2.0

sigma_0, sigma_z = np.eye(2), np.array([[1, 0], [0, -1]])
sigma_x, sigma_y = np.array([[0, 1], [1, 0]]), np.array([[0, -1j], [1j, 0]])
tau_0, tau_x, tau_y, tau_z = sigma_0, sigma_x, sigma_y, sigma_z

tz_s0 = np.kron(tau_z, sigma_0)
t0_sz = np.kron(tau_0, sigma_z)
tx_s0 = np.kron(tau_x, sigma_0)
tz_sy = np.kron(tau_z, sigma_y)


def build_BdG(Vz):
    H = np.zeros((4 * N, 4 * N), dtype=complex)
    Hs = -mu * tz_s0 + Vz * t0_sz + delta * tx_s0
    Hh = -t * tz_s0 - 1j * (alpha / 2.0) * tz_sy

    for i in range(N):
        H[4 * i : 4 * i + 4, 4 * i : 4 * i + 4] = Hs
        if i < N - 1:
            H[4 * i : 4 * i + 4, 4 * (i + 1) : 4 * (i + 1) + 4] = Hh
            H[4 * (i + 1) : 4 * (i + 1) + 4, 4 * i : 4 * i + 4] = Hh.conj().T
    return H


def get_zero_mode_density(Vz):
    H = build_BdG(Vz)
    evals, evecs = np.linalg.eigh(H)

    # Índice del autovalor más cercano a E = 0
    idx_zero = np.argmin(np.abs(evals))
    psi_zero = evecs[:, idx_zero]

    # Densidad de probabilidad por sitio (suma sobre componentes de Nambu)
    density = np.zeros(N)
    for i in range(N):
        psi_site = psi_zero[4 * i : 4 * i + 4]
        density[i] = np.sum(np.abs(psi_site) ** 2)

    return density, evals[idx_zero]


# Cálculo para fase trivial y topológica
prob_trivial, E_triv = get_zero_mode_density(0.5)
prob_topo, E_topo = get_zero_mode_density(2.0)

# Gráfica
sitios = np.arange(1, N + 1)
plt.figure(figsize=(9, 4.5))
plt.plot(
    sitios,
    prob_trivial,
    "o--",
    color="darkorange",
    label=f"Fase Trivial ($V_z = 0.5$, $E \\approx {E_triv:.3f}$)",
)
plt.plot(
    sitios,
    prob_topo,
    "s-",
    color="navy",
    label=f"Fase Topológica ($V_z = 2.0$, $E \\approx {E_topo:.5f}$)",
)

plt.xlabel("Sitio de la cadena ($i$)")
plt.ylabel(r"Densidad de probabilidad $|\psi_i|^2$")
plt.title(
    "Localización espacial del modo de menor energía (Prueba de Majorana)"
)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

: 

Transporte cuántico y conductancia $G(V)$ con Kwant.

Para comparar los resultados teóricos con los experimentos de laboratorio, los códigos de transporte calculan la corrientre eléctrica y la conductancia diferencial en uniones entre un metal "normal" y un superconductor (NS). Se simula la reflexión de Andreev resonante en la interfaz entre el electrodo normal y el hilo superconductor topológico. Para implementar el código se emplean funciones de Green de superficie o matrices de dispersión (scattering matrices) para calcular la probabilidad de reflexión de electrones en huecos (reflexión de Andreev) a partir de la formula de Blonder-Tinkham-Klapwjik o extensiones multipolares. Se deriva numéricamente la intensidad con respecto al voltaje para obtener la conductancia diferencial $\frac{dI}{dV}$. Este código busca generar mapas de color o curvas donde se aprecie la aparición de una anomalía a sesgo cero (Zero Bias anomaly, ZBA) cuantificada en G = \frac{2e^2}{h} (en el límite ideal) justo en el centro de la brecha superconductora cuando se enciende el campo magnético topológico.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import kwant
import matplotlib.pyplot as plt

# 1. Definir matrices
sigma_0, sigma_z = np.eye(2), np.array([[1, 0], [0, -1]])
sigma_x, sigma_y = np.array([[0, 1], [1, 0]]), np.array([[0, -1j], [1j, 0]])
tau_0, tau_x, tau_y, tau_z = sigma_0, sigma_x, sigma_y, sigma_z

tz_s0 = np.kron(tau_z, sigma_0)
t0_sz = np.kron(tau_0, sigma_z)
tx_s0 = np.kron(tau_x, sigma_0)
tz_sy = np.kron(tau_z, sigma_y)

# 2. Funciones del sistema
def onsite(site, p):
    return -p.mu * tz_s0 + p.Vz * t0_sz + p.delta * tx_s0

def hopping(site1, site2, p):
    return -p.t * tz_s0 - 1j * (p.alpha / 2.0) * tz_sy

# 3. Construcción del sistema
def make_system(L=50):
    lat = kwant.lattice.chain(norbs=4)
    syst = kwant.Builder()
    syst[lat.shape(lambda pos: 0 <= pos[0] < L, (0,))] = onsite
    syst[lat.neighbors()] = hopping
    
    lead = kwant.Builder(kwant.TranslationalSymmetry([-1]))
    lead[lat(0)] = -2.0 * tz_s0
    lead[lat.neighbors()] = -1.0 * tz_s0
    
    syst.attach_lead(lead)
    return syst.finalized()

# 4. Probar creación
syst = make_system(L=50)
print("¡Sistema construido con éxito! Número de sitios:", syst.graph.num_nodes)

: 

In [3]:
import kwant
print("Kwant versión:", kwant.__version__)

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import kwant
import matplotlib.pyplot as plt

# --- 1. Definición de matrices de Pauli y Nambu ---
sigma_0, sigma_z = np.eye(2), np.array([[1, 0], [0, -1]])
sigma_x, sigma_y = np.array([[0, 1], [1, 0]]), np.array([[0, -1j], [1j, 0]])
tau_0, tau_x, tau_y, tau_z = sigma_0, sigma_x, sigma_y, sigma_z

tz_s0 = np.kron(tau_z, sigma_0)
t0_sz = np.kron(tau_0, sigma_z)
tx_s0 = np.kron(tau_x, sigma_0)
tz_sy = np.kron(tau_z, sigma_y)


# --- 2. Funciones del Hamiltoniano local y de salto ---
def onsite(site, p):
    return -p.mu * tz_s0 + p.Vz * t0_sz + p.delta * tx_s0


def hopping(site1, site2, p):
    return -p.t * tz_s0 - 1j * (p.alpha / 2.0) * tz_sy


# --- 3. Construcción del sistema cuántico con Kwant ---
def make_system(L=50):
    lat = kwant.lattice.chain(norbs=4)
    syst = kwant.Builder()

    # Añadir sitios y enlaces de la cadena superconductora
    syst[lat.shape(lambda pos: 0 <= pos[0] < L, (0,))] = onsite
    syst[lat.neighbors()] = hopping

    # Definir el Lead Normal metálico acoplado a la izquierda (x = 0)
    lead = kwant.Builder(kwant.TranslationalSymmetry([-1]))
    lead[lat(0)] = -2.0 * tz_s0
    lead[lat.neighbors()] = -1.0 * tz_s0

    syst.attach_lead(lead)
    return syst.finalized()


# --- 4. Objeto de parámetros del sistema ---
class Params:

    def __init__(self, Vz):
        self.t = 1.0
        self.mu = -2.0
        self.delta = 1.0
        self.alpha = 2.0
        self.Vz = Vz


syst = make_system(L=50)
energies = np.linspace(-0.5, 0.5, 100)

conductance_triv = []
conductance_topo = []

params_triv = Params(Vz=0.5)
params_topo = Params(Vz=2.0)

# --- 5. Cálculo de la conductancia BTK mediante matriz de dispersión ---
for E in energies:
    # Fase trivial (Vz = 0.5)
    smat_triv = kwant.smatrix(syst, energy=E, args=[params_triv])
    s_he_triv = smat_triv.submatrix((0, 1), (0, 0))
    conductance_triv.append(2.0 * np.sum(np.abs(s_he_triv) ** 2))

    # Fase topológica (Vz = 2.0)
    smat_topo = kwant.smatrix(syst, energy=E, args=[params_topo])
    s_he_topo = smat_topo.submatrix((0, 1), (0, 0))
    conductance_topo.append(2.0 * np.sum(np.abs(s_he_topo) ** 2))

# --- 6. Gráfica de la Conductancia G vs V ---
plt.figure(figsize=(8, 5))
plt.plot(
    energies,
    conductance_triv,
    "--",
    color="darkorange",
    linewidth=2,
    label="Fase Trivial ($V_z = 0.5$)",
)
plt.plot(
    energies,
    conductance_topo,
    "-",
    color="navy",
    linewidth=2,
    label="Fase Topológica ($V_z = 2.0$)",
)

plt.axhline(
    y=2.0,
    color="red",
    linestyle=":",
    label=r"Cuantización $G = 2e^2/h$ (ZBCP)",
)
plt.xlabel("Energía de inyección / Voltaje $eV$")
plt.ylabel(r"Conductancia $G$ [$2e^2/h$]")
plt.title("Firma de Majorana: Pico de Conductancia a Voltaje Cero (ZBCP)")
plt.ylim(-0.1, 2.3)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

: 